In [113]:
import pandas as pd
import numpy as np

In [114]:
df = pd.read_csv("../../data/car_fuel_efficiency_2026.csv")

In [115]:
df = df[[
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
    ]]


In [116]:
df.horsepower.median()

np.float64(254.0)

In [117]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

X_train = df.iloc[idx[:n_train]]
X_val = df.iloc[idx[n_train:n_train + n_val]]
X_test = df.iloc[idx[n_train + n_val:]]

In [ ]:
y_train = X_train.fuel_efficiency_mpg
y_val = X_val.fuel_efficiency_mpg
y_test = X_test.fuel_efficiency_mpg
for part in [X_train, X_val, X_test]:
    del part['fuel_efficiency_mpg']

In [119]:
features= [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year']


In [ ]:
X_train.horsepower = X_train.horsepower.fillna(0)
#mean_hp = X_train['horsepower'].mean()
#X_train['horsepower'] = X_train['horsepower'].fillna(mean_hp)

In [122]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    w_full = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w_full[0], w_full[1:]

In [123]:
w0, w = train_linear_regression(X_train, y_train)

In [124]:
w0, w

(np.float64(-153.8849618823138),
 array([-1.51505520e-03, -4.74024794e-06, -3.95418397e-03,  1.02146785e-01]))

In [125]:
X_val_zero = X_val.fillna(0).to_numpy()
y_pred = w0 + X_val_zero.dot(w)
# X_val_mean = X_val.fillna(X_val.mean()).to_numpy()
# y_pred = w0 + X_val_mean.dot(w)
def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

In [126]:
y_pred.shape, y_val.shape

((2000,), (2000,))

In [127]:
score_zero = rmse(y_val, y_pred)
round(score_zero, 3)

np.float64(2.205)

In [128]:
# Regularized linear with regression
def train_linear_regression_reg(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    w_full = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w_full[0], w_full[1:]

In [129]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

In [135]:
scores = {}

for r in r_values:
    w0, w = train_linear_regression_reg(X_train, y_train, r)
    y_pred = w0 + X_val.dot(w)
    scores = round(rmse(y_val, y_pred), 4)
    print(scores)

2.1896
2.1903
2.2102
2.3413
2.4038
2.4143
2.4244


In [139]:

def split_data(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)
    return df_train, df_val, df_test

In [ ]:
scores = []

for seed in range(10):
    X_train, X_val, X_test = split_data(df, seed)

    y_train = X_train.fuel_efficiency_mpg
    y_val = X_val.fuel_efficiency_mpg

    del X_train['fuel_efficiency_mpg']
    del X_val['fuel_efficiency_mpg']

    X_train.horsepower = X_train.horsepower.fillna(0)
    X_val.horsepower = X_val.horsepower.fillna(0)

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    scores.append(rmse(y_val, y_pred))

std = round(np.std(scores), 3)
std

    

np.float64(0.029)

In [156]:
X_train, X_val, X_test = split_data(df, seed=9)
X_train = pd.concat([X_train, X_val], ignore_index=True)

y_train = X_train.fuel_efficiency_mpg.values
y_test = X_test.fuel_efficiency_mpg.values

del X_train['fuel_efficiency_mpg']
del X_test['fuel_efficiency_mpg']

X_train.horsepower = X_train.horsepower.fillna(0)
X_test.horsepower = X_test.horsepower.fillna(0)

w0, w = train_linear_regression_reg(X_train, y_train, 0.001)
y_pred = w0 + X_test.dot(w)
score_test = rmse(y_test, y_pred)
score_test

np.float64(2.235827747191731)